### item availability with details Tool on warehouse

In [ ]:
def check_warehouse_availability(items: list[dict]) -> dict:

    """Check availability of items across warehouses, including partial fulfillment options.
    
    Args:
        items: A list of items to check. Each item is a dictionary with keys: product_id, quantity.
        
    Returns:
        A dictionary containing:
        - can_fulfill_completely: bool indicating if all items can be fulfilled from at least one warehouse
        - warehouses_full_fulfillment: list of warehouses that can fulfill the entire order
        - warehouses_partial_fulfillment: list of warehouses with partial availability
        - unavailable_items: list of items that cannot be fulfilled from any warehouse
        - details: detailed breakdown per warehouse with availability for each item
    """
    # Connect to the Postgres database (assuming local or environment-provided connection)
    # You may need to set these credentials appropriately for your platform/environment
    PG_CONN_INFO = {
        "dbname": "tools_database",
        "user": "langgraph_user",
        "password": "langgraph_password",
        "host": "localhost",
        "port": 5432
    }

    # We'll use psycopg2 to query the warehouse inventory table
    try:
        conn = psycopg2.connect(**PG_CONN_INFO)
    except Exception as e:
        # Fail gracefully
        return {"error": f"Database connection failed: {e}"}
    
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            # Query inventory for all product/warehouse info in one go
            all_product_ids = [item["product_id"] for item in items]
            sql = """
                SELECT warehouse_id, product_id, available_quantity, warehouse_location, warehouse_name
                FROM warehouses.inventory
                WHERE product_id = ANY(%s)
            """
            cur.execute(sql, (all_product_ids,))
            inventory_rows = cur.fetchall()
    except Exception as e:
        conn.close()
        return {"error": f"Inventory query failed: {e}"}
    
    # Map (warehouse_id) -> {product_id: (available_quantity, info...)}
    warehouse_map = {}
    for row in inventory_rows:
        wid = row["warehouse_id"]
        pid = row["product_id"]
        avail = row["available_quantity"]
        loc = row.get("warehouse_location", None)
        name = row.get("warehouse_name", None)
        if wid not in warehouse_map:
            warehouse_map[wid] = {
                "items": {},
                "warehouse_location": loc,
                "warehouse_name": name,
            }
        warehouse_map[wid]["items"][pid] = avail
    
    # Now calculate which warehouses can fully or partially fulfill the order
    warehouses_full_fulfillment = []
    warehouses_partial_fulfillment = []
    details = {}
    unavailable_items = []
    fully_unavailable_products = set()
    
    # For each warehouse, check how many requested items can be fulfilled
    for wid, wdata in warehouse_map.items():
        can_fulfill_all = True
        partial_fulfillment = False
        warehouse_detail = {
            "warehouse_id": wid,
            "warehouse_location": wdata["warehouse_location"],
            "warehouse_name": wdata["warehouse_name"],
            "items": []
        }
        
        for item in items:
            pid = item["product_id"]
            qty = item["quantity"]
            available = wdata["items"].get(pid, 0)
            
            if available >= qty:
                item_status = {
                    "product_id": pid,
                    "requested_quantity": qty,
                    "available_quantity": available,
                    "status": "available"
                }
            elif available > 0:
                item_status = {
                    "product_id": pid,
                    "requested_quantity": qty,
                    "available_quantity": available,
                    "status": "partial"
                }
                can_fulfill_all = False
                partial_fulfillment = True
            else:
                item_status = {
                    "product_id": pid,
                    "requested_quantity": qty,
                    "available_quantity": 0,
                    "status": "unavailable"
                }
                can_fulfill_all = False
                partial_fulfillment = True
            warehouse_detail["items"].append(item_status)
        
        details[wid] = warehouse_detail
        if can_fulfill_all:
            warehouses_full_fulfillment.append(wid)
        elif partial_fulfillment:
            warehouses_partial_fulfillment.append(wid)
    
    # Determine items that are completely unavailable in any warehouse
    for item in items:
        pid = item["product_id"]
        qty = item["quantity"]
        found = False
        for wdata in warehouse_map.values():
            if wdata["items"].get(pid, 0) >= qty:
                found = True
                break
        if not found:
            # No warehouse can fulfill the desired quantity for this item
            # But maybe some have partial -- let's check if it's completely missing
            max_anywhere = max([wdata["items"].get(pid, 0) for wdata in warehouse_map.values()], default=0)
            if max_anywhere == 0:
                fully_unavailable_products.add(pid)
            unavailable_items.append({
                "product_id": pid,
                "requested_quantity": qty,
                "max_available_quantity": max_anywhere,
                "status": "fully_unavailable" if max_anywhere == 0 else "partially_available"
            })

    conn.close()

    result = {
        "can_fulfill_completely": (len(warehouses_full_fulfillment) > 0),
        "warehouses_full_fulfillment": warehouses_full_fulfillment,
        "warehouses_partial_fulfillment": warehouses_partial_fulfillment,
        "unavailable_items": unavailable_items,
        "details": details
    }
    return result

In [ ]:
shopping_cart = [
    {
        "product_id": "B0BM963XNT",
        "quantity": 3
    },
    {
        "product_id": "B0BNKJDF99",
        "quantity": 2
    },
    {
        "product_id": "B0BVZ47N6N",
        "quantity": 5
    }
]

In [ ]:
availability = check_warehouse_availability(shopping_cart)

In [ ]:
availability

In [ ]:
shopping_cart = [
    {
        "product_id": "B0BM963XNT",
        "quantity": 10
    },
    {
        "product_id": "B0BNKJDF99",
        "quantity": 10
    },
    {
        "product_id": "B0BVZ47N6N",
        "quantity": 10
    }
]

In [ ]:
availability = check_warehouse_availability(shopping_cart)
availability

### Item Reservation Tool

In [ ]:
def reserve_warehouse_items(reservations: list[dict]) -> dict:
    # For each reservation, we should add the reserved quantity to the existing reserved_quantity
    # if there is enough available_quantity. We must NOT just replace it with the new value.

    # (The main update is to ensure the reserved_quantity is incremented, not simply set.)

    """Reserve items from multiple warehouses in a single transaction.
    
    Args:
        reservations: A list of reservations. Each reservation is a dictionary with keys:
                     - warehouse_id: The warehouse to reserve from
                     - product_id: The product to reserve
                     - quantity: The quantity to reserve
        
    Returns:
        A dictionary containing:
        - success: bool indicating if all reservations were successful
        - reserved_items: list of successfully reserved items
        - failed_items: list of items that could not be reserved
    """
    import psycopg2

    # Example: assumes a psycopg2 connect function and a PostgreSQL DB setup with the warehouse schema
    def get_db_conn():
        # In real code, replace with your connection config/logic
        return psycopg2.connect(
            dbname="tools_database",
            user="langgraph_user",
            password="langgraph_password",
            host="localhost",
            port=5432
        )

    success = True
    reserved_items = []
    failed_items = []

    conn = None
    try:
        conn = get_db_conn()
        conn.autocommit = False
        cur = conn.cursor()

        for reservation in reservations:
            warehouse_id = reservation["warehouse_id"]
            product_id = reservation["product_id"]
            quantity = reservation["quantity"]

            # Check current available_quantity in a transaction (FOR UPDATE to lock row)
            cur.execute(
                """
                SELECT id, available_quantity, reserved_quantity, total_quantity
                FROM warehouses.inventory
                WHERE warehouse_id=%s AND product_id=%s
                FOR UPDATE
                """,
                (warehouse_id, product_id)
            )
            row = cur.fetchone()
            if row is None:
                failed_items.append({
                    "warehouse_id": warehouse_id,
                    "product_id": product_id,
                    "requested_quantity": quantity,
                    "reason": "Item not found"
                })
                success = False
                continue

            inv_id, available_quantity, reserved_quantity, total_quantity = row

            if available_quantity >= quantity:
                cur.execute(
                    """
                    UPDATE warehouses.inventory
                    SET reserved_quantity = reserved_quantity + %s
                    WHERE id=%s
                    RETURNING reserved_quantity, available_quantity
                    """,
                    (quantity, inv_id)
                )
                updated = cur.fetchone()
                reserved_items.append({
                    "warehouse_id": warehouse_id,
                    "product_id": product_id,
                    "reserved_quantity": quantity,
                    "total_reserved": updated[0],
                    "remaining_available": updated[1],
                })
            else:
                failed_items.append({
                    "warehouse_id": warehouse_id,
                    "product_id": product_id,
                    "requested_quantity": quantity,
                    "available_quantity": available_quantity,
                    "reason": "Not enough available quantity"
                })
                success = False

        if success:
            conn.commit()
        else:
            conn.rollback()
            reserved_items = []
    except Exception as e:
        if conn:
            conn.rollback()
        success = False
        reserved_items = []
        failed_items.append({"error": str(e)})
    finally:
        if conn:
            conn.close()

    return {
        "success": success,
        "reserved_items": reserved_items,
        "failed_items": failed_items
    }

In [ ]:
reservations = [
    {
        "product_id": "B08MFBYDN2",
        "quantity": 3,
        "warehouse_id": "IN-DEL-01"
    },
    {
        "product_id": "B0C272636P",
        "quantity": 2,
        "warehouse_id": "IN-BLR-01"
    },
    {
        "product_id": "B09Q36C75R",
        "quantity": 5,
        "warehouse_id": "IN-KOL-01"
    }
]

In [ ]:
result  = reserve_warehouse_items(reservations)
result

In [ ]:
reservations = [
    {
        "product_id": "B08MFBYDN2",
        "quantity": 3,
        "warehouse_id": "IN-DEL-01"
    },
    {
        "product_id": "B0C272636P",
        "quantity": 2,
        "warehouse_id": "IN-BLR-01"
    },
    {
        "product_id": "B09Q36C75R908",
        "quantity": 5,
        "warehouse_id": "IN-KOL-01"
    }
]

In [ ]:
result  = reserve_warehouse_items(reservations)
result